# Tema de Programare: Analiza Distribuțiilor


Bine ai venit la tema despre **Analiza Distribuțiilor**!

Înțelegerea formei datelor tale este unul dintre primii — și cei mai importanți — pași în orice proiect de machine learning sau data science. Analiza distribuțiilor răspunde la întrebări fundamentale precum:

| Întrebare | Instrument |
|----------|------|
| Care este valoarea tipică? | Mean / Median |
| Cât de răspândite sunt datele? | Std / IQR |
| Este distribuția simetrică? | Skewness |
| Există cozi neobișnuit de grele? | Kurtosis |
| Unde sunt outlierii? | Box plot |

> **Pont:** Uită-te întotdeauna atât la **histogramă** (forma), cât și la **box plot** (outlierii).
> Ele scot la iveală lucruri diferite — o histogramă poate ascunde valori extreme pe care box plot-ul
> le semnalează clar ca outlieri.

**Ce vei face în această temă:**

* Calculezi un set complet de statistici descriptive pentru coloane individuale
* Clasifici distribuțiile după profilul lor de asimetrie și kurtoză
* Construiești o vizualizare cu 3 panouri: histogramă, KDE și box plot
* Rezumi statisticile de dispersie pentru toate coloanele numerice dintr-un foc
* Marchezi automat coloanele a căror asimetrie depășește un prag configurabil

Să intrăm în subiect!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu excepția celor în care trebuie să trimiți soluțiile sau când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga și nu modifica niciun cod în afara acestor comentarii**.

* Poți adăuga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, așa că nu te baza pe celulele create de tine pentru codul soluției; folosește spațiile oferite în notebook.

---


## Cuprins
- [Importuri](#imports)
- [1 - Înțelegerea Datelor](#1---understanding-the-data)
- [2 - Statistici ale Distribuției](#2---distribution-statistics)
    - **[Exercițiul 1 - `compute_distribution_stats`](#exercise-1---compute_distribution_stats)**
- [3 - Clasificarea Distribuției](#3---distribution-classification)
    - **[Exercițiul 2 - `classify_distribution`](#exercise-2---classify_distribution)**
- [4 - Analiză Vizuală](#4---visual-analysis)
    - **[Exercițiul 3 - `plot_distribution_triple`](#exercise-3---plot_distribution_triple)**
- [5 - Tabel Rezumat Numeric](#5---numeric-summary-table)
    - **[Exercițiul 4 - `summarize_all_numeric`](#exercise-4---summarize_all_numeric)**
- [6 - Detectarea Asimetriei](#6---skewness-detection)
    - **[Exercițiul 5 - `detect_skewed_columns`](#exercise-5---detect_skewed_columns)**
- [7 - Punem Totul Cap la Cap](#7---putting-it-all-together)


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib


In [ ]:
import helper_utils
import unittests


<a name='1---understanding-the-data'></a>
## 1 - Înțelegerea Datelor

Vom lucra cu un set de date simulat cu **comenzi e-commerce** (300 de rânduri). Fiecare rând
reprezintă o comandă a unui client și conține atât caracteristici numerice, cât și categoriale.
Coloanele numerice acoperă mai multe forme de distribuție — ceea ce face acest
set de date ideal pentru a exersa analiza distribuțiilor.


In [ ]:
df = helper_utils.get_orders_df()
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()


In [ ]:
print(df.describe().round(2))


<a name='2---distribution-statistics'></a>
## 2 - Statistici ale Distribuției

<a name='exercise-1---compute_distribution_stats'></a>
### **Exercițiul 1 - `compute_distribution_stats`**

**Sarcina ta:**
Calculează un dicționar cu statistici descriptive-cheie pentru o singură coloană numerică.

**Cerințe:**
- Acceptă un DataFrame `df` și un nume de coloană `column` (șir)
- Returnează un `dict` cu exact aceste șase chei: `mean`, `median`, `std`, `iqr`, `skewness`, `kurtosis`
- `iqr` este intervalul intercuartilic: Q3 (0.75) minus Q1 (0.25)
- `skewness` și `kurtosis` folosesc definițiile Fisher din pandas (excess kurtosis)

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde)</font></b></summary>

  **Ghid pas cu pas:**

  1. Extrage coloana ca Series: `series = df[column]`
  2. Folosește `series.mean()`, `series.median()`, `series.std()` pentru primele trei valori.
  3. Pentru `iqr`: `series.quantile(0.75) - series.quantile(0.25)`
  4. Pentru `skewness`: `series.skew()`
  5. Pentru `kurtosis` (exces / Fisher): `series.kurt()`
  6. Returnează toate cele șase valori într-un dicționar cu numele exacte ale cheilor prezentate mai sus.

</details>


In [ ]:
# GRADED FUNCTION: compute_distribution_stats
def compute_distribution_stats(df, column):
    """
    Calculează statistici descriptive pentru o singură coloană numerică.

    Args:
        df (pd.DataFrame): Sursa datelor.
        column (str): Numele coloanei numerice.

    Returns:
        dict: Chei: mean, median, std, iqr, skewness, kurtosis.
    """
    ### ÎNCEPUT DE COD AICI ###
    # SCRIE CODUL TĂU AICI
    raise NotImplementedError("Implementează această funcție")
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Demo: inspectează statisticile pentru coloana 'order_value'
stats = compute_distribution_stats(df, 'order_value')
print("Statistici ale distribuției pentru 'order_value':")
for key, value in stats.items():
    print(f"  {key:<12}: {value:.4f}")


In [ ]:
# Testează codul tău!
unittests.exercise_1(compute_distribution_stats)


<a name='3---distribution-classification'></a>
## 3 - Clasificarea Distribuției

<a name='exercise-2---classify_distribution'></a>
### **Exercițiul 2 - `classify_distribution`**

**Sarcina ta:**
Având asimetria și excesul de kurtoză al unei distribuții, returnează un șir
care descrie forma sa.

**Cerințe — folosește EXACT aceste praguri:**

| Condiție | Etichetă |
|-----------|-------|
| `\|skewness\| < 0.5` AND `\|kurtosis\| < 1` | `"symmetric"` |
| `0.5 <= skewness < 1` | `"moderate_right_skew"` |
| `skewness >= 1` | `"strong_right_skew"` |
| `-1 < skewness <= -0.5` | `"moderate_left_skew"` |
| `skewness <= -1` | `"strong_left_skew"` |
| În caz contrar | `"symmetric"` |

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde)</font></b></summary>

  **Ghid pas cu pas:**

  1. Verifică mai întâi asimetria puternică la dreapta (`skewness >= 1`), apoi pe cea moderată la dreapta.
  2. Verifică asimetria puternică la stânga (`skewness <= -1`), apoi pe cea moderată la stânga.
  3. Un simplu `else` la final acoperă cazul simetric.
  4. Observă că parametrul `kurtosis` este disponibil pentru verificarea simetriei,
     dar în practică pragurile de asimetrie au prioritate.

</details>


In [ ]:
# GRADED FUNCTION: classify_distribution
def classify_distribution(skewness, kurtosis):
    """
    Clasifică o distribuție pe baza asimetriei și kurtozei sale.

    Args:
        skewness (float): Valoarea asimetriei distribuției.
        kurtosis (float): Excesul de kurtoză (definiția Fisher).

    Returns:
        str: Una dintre valorile: "symmetric", "moderate_right_skew", "strong_right_skew",
             "moderate_left_skew", "strong_left_skew".
    """
    ### ÎNCEPUT DE COD AICI ###
    # SCRIE CODUL TĂU AICI
    raise NotImplementedError("Implementează această funcție")
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Demo: clasifică fiecare coloană numerică
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print(f"{'Coloană':<30} {'Asimetrie':>10} {'Kurtoză':>10}  Clasificare")
print("-" * 70)
for col in numeric_cols:
    s = compute_distribution_stats(df, col)
    label = classify_distribution(s['skewness'], s['kurtosis'])
    print(f"{col:<30} {s['skewness']:>10.3f} {s['kurtosis']:>10.3f}  {label}")


In [ ]:
# Testează codul tău!
unittests.exercise_2(classify_distribution)


<a name='4---visual-analysis'></a>
## 4 - Analiză Vizuală

<a name='exercise-3---plot_distribution_triple'></a>
### **Exercițiul 3 - `plot_distribution_triple`**

**Sarcina ta:**
Creează o figură 1×3 care arată trei perspective complementare asupra unei coloane numerice:
o histogramă, un KDE (kernel density estimate) și un box plot.

**Cerințe:**
- Returnează un tuplu `(fig, axes)` unde `axes` are exact 3 elemente
- **Panoul 0 — Histogramă:** 30 bin-uri, `edgecolor='white'`
- **Panoul 1 — KDE:** folosește `df[column].plot.kde(ax=axes[1])`
- **Panoul 2 — Box plot:** `axes[2].boxplot(df[column].dropna(), vert=True)`
- Fiecare subplot trebuie să aibă un titlu descriptiv și etichete pentru axe
- Apelează `plt.tight_layout()` înainte de return

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde)</font></b></summary>

  **Ghid pas cu pas:**

  1. Creează figura: `fig, axes = plt.subplots(1, 3, figsize=figsize)`
  2. Elimină valorile NaN: `series = df[column].dropna()`
  3. Histogramă: `axes[0].hist(series, bins=30, edgecolor='white')`
     - Setează titlul cu `axes[0].set_title(...)` și eticheta cu `axes[0].set_xlabel(...)`
  4. KDE: `series.plot.kde(ax=axes[1])`
     - Adaugă un titlu și eticheta axei x
  5. Box plot: `axes[2].boxplot(series, vert=True)`
     - Adaugă un titlu și eticheta axei y
  6. `plt.tight_layout()` și apoi `return fig, axes`

</details>


In [ ]:
# GRADED FUNCTION: plot_distribution_triple
def plot_distribution_triple(df, column, figsize=(14, 4)):
    """
    Creează o figură cu 3 panouri: histogramă, KDE și box plot.

    Args:
        df (pd.DataFrame): Sursa datelor.
        column (str): Coloana numerică ce trebuie vizualizată.
        figsize (tuple): Dimensiunile figurii.

    Returns:
        tuple: (fig, axes), unde axes este un array cu 3 obiecte Axes.
    """
    ### ÎNCEPUT DE COD AICI ###
    # SCRIE CODUL TĂU AICI
    raise NotImplementedError("Implementează această funcție")
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Demo: vizualizează două coloane una după alta
fig1, _ = plot_distribution_triple(df, 'order_value')
plt.show()

fig2, _ = plot_distribution_triple(df, 'customer_age')
plt.show()


In [ ]:
# Testează codul tău!
unittests.exercise_3(plot_distribution_triple)


<a name='5---numeric-summary-table'></a>
## 5 - Tabel Rezumat Numeric

<a name='exercise-4---summarize_all_numeric'></a>
### **Exercițiul 4 - `summarize_all_numeric`**

**Sarcina ta:**
Construiește un DataFrame rezumat al dispersiei care acoperă **toate coloanele numerice** dintr-un singur apel.

**Cerințe:**
- Ignoră coloanele cu dtype `object` sau `category` (folosește `df.select_dtypes(include='number')`)
- Returnează un `pd.DataFrame` indexat după numele coloanei
- Coloanele DataFrame-ului returnat: `std`, `iqr`, `skewness`, `kurtosis`
- Toate valorile rotunjite la 3 zecimale

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde)</font></b></summary>

  **Ghid pas cu pas:**

  1. Obține coloanele numerice: `numeric_cols = df.select_dtypes(include='number').columns.tolist()`
  2. Construiește un dicționar `rows` unde fiecare cheie este numele unei coloane și valoarea este un sub-dicționar
     cu cheile `std`, `iqr`, `skewness`, `kurtosis`.
  3. Pentru fiecare coloană `col`, calculează:
     - `round(df[col].std(), 3)`, `round(df[col].quantile(0.75) - df[col].quantile(0.25), 3)`,
       `round(df[col].skew(), 3)`, `round(df[col].kurt(), 3)`
  4. Convertește în DataFrame: `pd.DataFrame(rows).T`
     (transpunerea `.T` pune coloanele ca rânduri și statisticile ca coloane)

</details>


In [ ]:
# GRADED FUNCTION: summarize_all_numeric
def summarize_all_numeric(df):
    """
    Construiește un tabel rezumat al dispersiei pentru toate coloanele numerice.

    Args:
        df (pd.DataFrame): Sursa datelor.

    Returns:
        pd.DataFrame: Index = numele coloanelor. Coloane: std, iqr, skewness, kurtosis.
    """
    ### ÎNCEPUT DE COD AICI ###
    # SCRIE CODUL TĂU AICI
    raise NotImplementedError("Implementează această funcție")
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Demo: afișează tabelul rezumat al dispersiei
summary = summarize_all_numeric(df)
print("Rezumatul dispersiei pentru toate coloanele numerice:")
print(summary.to_string())
print("\nTipare remarcabile:")
print("  - order_value: std mare și IQR mare → dispersie largă")
print("  - days_to_ship: aproape uniformă, deci asimetrie apropiată de 0")
print("  - days_since_last_purchase: lognormală → ne așteptăm la asimetrie la dreapta")


In [ ]:
# Testează codul tău!
unittests.exercise_4(summarize_all_numeric)


<a name='6---skewness-detection'></a>
## 6 - Detectarea Asimetriei

<a name='exercise-5---detect_skewed_columns'></a>
### **Exercițiul 5 - `detect_skewed_columns`**

**Sarcina ta:**
Returnează o listă sortată de nume de coloane a căror asimetrie absolută atinge sau depășește
pragul dat.

**Cerințe:**
- Ia în calcul doar coloanele numerice (float/int) — folosește `df.select_dtypes(include='number')`
- Include o coloană dacă `abs(skewness) >= threshold`
- Pragul implicit este `1.0`
- Returnează lista sortată alfabetic

> **De ce contează:** Caracteristicile cu asimetrie mare necesită adesea un log-transform
> (`np.log1p`) înainte de a fi introduse în modele liniare sau algoritmi bazați pe distanță.
> Automatizarea detecției asimetriei te ajută să construiești pipeline-uri robuste de preprocesare.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (apasă pentru a extinde)</font></b></summary>

  **Ghid pas cu pas:**

  1. Obține coloanele numerice: `numeric_cols = df.select_dtypes(include='number').columns`
  2. Folosește un list comprehension: `[col for col in numeric_cols if abs(df[col].skew()) >= threshold]`
  3. Învelește rezultatul cu `sorted(...)` pentru a returna numele în ordine alfabetică.

</details>


In [ ]:
# GRADED FUNCTION: detect_skewed_columns
def detect_skewed_columns(df, threshold=1.0):
    """
    Identifică coloanele numerice cu asimetrie semnificativă.

    Args:
        df (pd.DataFrame): Sursa datelor.
        threshold (float): Pragul absolut al asimetriei (implicit 1.0).

    Returns:
        list[str]: Listă sortată de nume de coloane cu |skewness| >= threshold.
    """
    ### ÎNCEPUT DE COD AICI ###
    # SCRIE CODUL TĂU AICI
    raise NotImplementedError("Implementează această funcție")
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Demo: detectează coloanele asimetrice la două praguri
skewed_strict = detect_skewed_columns(df, threshold=1.0)
skewed_loose  = detect_skewed_columns(df, threshold=0.5)
print("Coloane cu |asimetrie| >= 1.0:", skewed_strict)
print("Coloane cu |asimetrie| >= 0.5:", skewed_loose)


In [ ]:
# Testează codul tău!
unittests.exercise_5(detect_skewed_columns)


<a name='7---putting-it-all-together'></a>
## 7 - Punem Totul Cap la Cap

Acum combinăm toate cele cinci funcții într-un dashboard complet de analiză a distribuțiilor.
Pentru fiecare coloană numerică, calculăm statistici, clasificăm forma distribuției
și generăm vizualizarea cu 3 panouri.


In [ ]:
# Dashboard complet pentru analiza distribuțiilor
import matplotlib
matplotlib.use('Agg')  # suprimă afișarea interactivă în timpul generării în masă

numeric_cols = df.select_dtypes(include='number').columns.tolist()

print(f"{'Coloană':<32} {'Mean':>8} {'Std':>8} {'IQR':>8} {'Asimetrie':>10} {'Formă'}")
print("-" * 80)

for col in numeric_cols:
    stats   = compute_distribution_stats(df, col)
    shape   = classify_distribution(stats['skewness'], stats['kurtosis'])
    print(
        f"{col:<32} {stats['mean']:>8.2f} {stats['std']:>8.2f} "
        f"{stats['iqr']:>8.2f} {stats['skewness']:>10.3f}  {shape}"
    )

print("\nSe generează vizualizările...")
for col in numeric_cols:
    fig, _ = plot_distribution_triple(df, col)
    plt.show()

skewed = detect_skewed_columns(df, threshold=1.0)
print(f"\nColoane recomandate pentru log-transform (|skew| >= 1): {skewed}")
